# Week 12 supplement · A genuine, bounded LLM-driven Agent
**Machine Learning in Engineering Science — NCKU + LAiMM**

A pretrained causal language model selects an action, a controller validates it, a local tool returns an observation, and that observation enters the next model prompt. This completes a bounded task with an evidence-backed numeric answer. The action sequence is not scripted. This is a constrained next-token policy, not an open-ended planner or benchmark of broad agent competence.

## 1. Two honestly different modes
- **Replay (default):** no model, key, GPU, network or API credit. Recompute local tool outputs using an authentic saved live decision trace; reject changed inputs or prompts. No new LLM inference occurs.
- **Live:** load the pinned 135M-parameter instruction-tuned checkpoint on CPU. The first run downloads approximately 270 MB of weights plus tokenizer files and, if necessary, Python packages. No paid API or API credit is used; allow time and bandwidth for installation/download.

The classroom fixture contains six **synthetic** temperatures in degrees Celsius. The threshold is a declared exercise value, not an engineering safety standard. Numeric results come from tools, not invented prose. Budget: at most four decisions; no repeated evidence tools. Completion requires the two requested observations. A terminal choice may have only one allowed action; that is a controller constraint, not a reasoning achievement.

Primary source: [SmolLM2-135M-Instruct model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct). The small model does not advertise the larger model’s native function-calling capability. We implement a transparent constrained-action interface ourselves.

In [ ]:
MODE='replay'  # Change to 'live' only when you want new model decisions.
TRACE_PATH='agent-live-trace.json'
from pathlib import Path
import json
# Bundled authentic live trace: replay works without a remote download.
SAVED_TRACE={'mode': 'live', 'model': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'model_revision': '12fd25f77366fa6b3b4b768ec3050bf629380bac', 'goal': 'Report the maximum temperature and the count of measurements at or above 80 degC. Use tools, then finish.', 'input_sha256': '5e2cec390c6cc9240aec2ef4c0f0e5c25677838a365b98ffb95f9fc13b7e756b', 'steps': [{'turn': 0, 'candidates': ['max_temperature', 'count_at_or_above', 'inspect_schema'], 'choice': 2, 'action': {'tool': 'inspect_schema', 'args': {}}, 'observation': {'case': 'synthetic packaging temperature inspection', 'unit': 'degC', 'rows': 6}, 'prompt_sha256': 'e5acabe1aa8bee25aa3efb29eb8b3685b9e29ed239e98b348a3a31dfc104936e', 'allowed_token_logits': [32.56756591796875, 32.11750793457031, 32.57199478149414]}, {'turn': 1, 'candidates': ['max_temperature', 'count_at_or_above'], 'choice': 1, 'action': {'tool': 'count_at_or_above', 'args': {'threshold': 80.0}}, 'observation': {'count': 3, 'threshold': 80.0, 'rows': 6}, 'prompt_sha256': '1a908b08119abc52bbd7effaf3d04db290d04ffe136b728cc099f03600003aca', 'allowed_token_logits': [32.31690979003906, 32.32471466064453]}, {'turn': 2, 'candidates': ['max_temperature'], 'choice': 0, 'action': {'tool': 'max_temperature', 'args': {}}, 'observation': {'maximum': 85.0, 'unit': 'degC'}, 'prompt_sha256': 'e8ed24bbf1b78acca07459d4429600ad1eb5d91dd31f5336a84458c354b0773d', 'allowed_token_logits': [32.450008392333984]}, {'turn': 3, 'candidates': ['finish'], 'choice': 0, 'action': {'tool': 'finish', 'args': {}}, 'observation': {'maximum': 85.0, 'count_at_or_above_80': 3}, 'prompt_sha256': 'b6bd4a1fd029de71d79d7401ed38b61862c6ddd67aac0005a0971deaf54c3ae5', 'allowed_token_logits': [31.83931541442871]}], 'final': {'maximum': 85.0, 'count_at_or_above_80': 3}, 'elapsed_seconds': 0.336, 'method': 'Pretrained causal-LM next-token logits select a constrained action letter; no hand-authored action sequence or paid API. Tools, not generated prose, supply final numeric fields.'}
Path(TRACE_PATH).write_text(json.dumps(SAVED_TRACE,indent=2))
print('Mode:',MODE)

## 2. Install only for live inference
Run this before importing Transformers. Replay skips installation entirely. A standard Colab CPU runtime already includes PyTorch; local users need a compatible PyTorch installation. If a runtime already imported conflicting package versions, restart and rerun from the top. Package versions below are the versions exercised locally, not a promise that every future Colab image is identical.

In [ ]:
if MODE=='live':
    import sys,subprocess,importlib.metadata as m
    pins={'transformers':'4.46.3','tokenizers':'0.20.3','huggingface-hub':'0.26.5'}
    def installed(k):
        try:return m.version(k)
        except m.PackageNotFoundError:return None
    missing=[k+'=='+v for k,v in pins.items() if installed(k)!=v]
    if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
    import torch
    print('CPU live inference; PyTorch',torch.__version__)
else:print('Replay: no package installation and no model inference.')

## 3. Inspect the full Agent before running
Read `run`: the model scores allowed action-letter tokens from the current prompt. Scores choose among the menu; they are **raw logits, not confidence probabilities or calibrated correctness**. The dispatcher accepts only exact predefined tools/arguments. Each observation changes the next prompt. The controller prevents premature completion and enforces the budget.

The final numeric object is assembled from checked observations. We do not claim the LLM generated those numbers or a free-form final explanation. All data and tool calls remain local. No email, external write or financial action is performed.

In [ ]:
"""A real pretrained LLM selects actions; pure local tools supply the numbers."""
from pathlib import Path
import json,hashlib,time
MODEL='HuggingFaceTB/SmolLM2-135M-Instruct'
REVISION='12fd25f77366fa6b3b4b768ec3050bf629380bac'
DATA={'case':'synthetic packaging temperature inspection','unit':'degC','temperature':[72.,81.,79.,85.,80.,74.]}
GOAL='Report the maximum temperature and the count of measurements at or above 80 degC. Use tools, then finish.'
ACTIONS={'max_temperature':{'tool':'max_temperature','args':{}},'count_at_or_above':{'tool':'count_at_or_above','args':{'threshold':80.0}},'inspect_schema':{'tool':'inspect_schema','args':{}},'finish':{'tool':'finish','args':{}}}
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()
def dispatch(a,data):
 if a not in list(ACTIONS.values()):raise ValueError('Unknown action or arguments')
 t=a['tool']
 if t=='max_temperature':return {'maximum':max(data['temperature']),'unit':data['unit']}
 if t=='count_at_or_above':return {'count':sum(v>=a['args']['threshold'] for v in data['temperature']),'threshold':80.0,'rows':len(data['temperature'])}
 if t=='inspect_schema':return {'case':data['case'],'unit':data['unit'],'rows':len(data['temperature'])}
 raise ValueError('unreachable')
def run(mode='replay',trace_path='agent-live-trace.json',data=DATA):
 if mode not in ['live','replay']:raise ValueError('mode must be live or replay')
 if data['unit']!='degC' or not data['temperature']:raise ValueError('Expected nonempty degC observations')
 tracefile=Path(trace_path)
 if mode=='live':
  import torch
  from transformers import AutoModelForCausalLM,AutoTokenizer
  torch.set_num_threads(4)
  tok=AutoTokenizer.from_pretrained(MODEL,revision=REVISION)
  llm=AutoModelForCausalLM.from_pretrained(MODEL,revision=REVISION).to('cpu').eval()
  token_ids=[tok.encode(c,add_special_tokens=False) for c in ['A','B','C']]
  assert all(len(t)==1 for t in token_ids),'Menu labels must each be a single token.'
 else:
  saved=json.loads(tracefile.read_text());assert saved['input_sha256']==digest(data) and saved['goal']==GOAL,'Replay input/goal differs; run live for new decisions.'
 observations={};steps=[];previous=[];start=time.time()
 for turn in range(4):
  ready=all(k in observations for k in ['max_temperature','count_at_or_above'])
  candidates=['finish'] if ready else [k for k in ['max_temperature','count_at_or_above','inspect_schema'] if k not in observations]
  prompt='You are a tool-using assistant. Choose the next permitted tool. Respond with one letter only.\nTask: '+GOAL+'\nKnown observations: '+json.dumps(observations)+'\nAvailable actions:\n'+'\n'.join(chr(65+i)+': '+json.dumps(ACTIONS[k]) for i,k in enumerate(candidates))+'\nNext action letter:'
  if mode=='live':
   formatted=tok.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
   encoded=tok(formatted,return_tensors='pt')
   with torch.inference_mode():
    logits=llm(**encoded).logits[0,-1];scores=[float(logits[token_ids[i][0]]) for i in range(len(candidates))]
   chosen=int(max(range(len(scores)),key=scores.__getitem__))
  else:
   old=saved['steps'][turn];assert old['prompt_sha256']==digest(prompt) and old['candidates']==candidates
   chosen=old['choice'];scores=old['allowed_token_logits']
  action=ACTIONS[candidates[chosen]]
  # Controller allows only known tools, validated fixed arguments and no repeated evidence calls.
  if action['tool']=='finish':
   assert ready
   observation={'maximum':observations['max_temperature']['maximum'],'count_at_or_above_80':observations['count_at_or_above']['count']}
  else:observation=dispatch(action,data);observations[action['tool']]=observation
  step={'turn':turn,'candidates':candidates,'choice':chosen,'action':action,'observation':observation,'prompt_sha256':digest(prompt),'allowed_token_logits':scores}
  if mode=='replay':assert step==saved['steps'][turn],'Replay tool observation differs'
  steps.append(step);print(mode.upper(),turn,action,'→',observation)
  if action['tool']=='finish':break
 else:raise RuntimeError('Budget exhausted without a supported answer')
 report={'mode':mode,'model':MODEL,'model_revision':REVISION,'goal':GOAL,'input_sha256':digest(data),'steps':steps,'final':observation,'elapsed_seconds':round(time.time()-start,3),'method':'Pretrained causal-LM next-token logits select a constrained action letter; no hand-authored action sequence or paid API. Tools, not generated prose, supply final numeric fields.'}
 assert report['final']=={'maximum':max(data['temperature']),'count_at_or_above_80':sum(v>=80 for v in data['temperature'])}
 if mode=='live':tracefile.write_text(json.dumps(report,indent=2))
 return report


## 4. Predict the route, then run the whole loop
Before execution, write which tool you expect first and why. Inspect the actual menu and selected action after each step. In replay, the recorded decisions are reused but tools execute again and their observations must match. Live inference overwrites the trace file with the new authentic run; the bundled source trace remains available above.

In [ ]:
prediction_before_run=''
report=run(MODE,TRACE_PATH)
assert report['final']=={'maximum':max(DATA['temperature']),'count_at_or_above_80':sum(v>=80 for v in DATA['temperature'])}
print('Supported final result:',report['final'])
print('Decisions:',len(report['steps']),'of maximum 4')

## 5. Test failure boundaries
These are controller tests, not model-reliability evidence. A changed input must invalidate the saved replay. An unknown tool must never dispatch. A unit mismatch must be rejected. After these checks, explain why a successful four-step trace does not prove robustness to a new task or malicious text.

In [ ]:
import copy
changed=copy.deepcopy(DATA);changed['temperature'][0]+=1.0
for label,call in [('changed replay input',lambda:run('replay',TRACE_PATH,changed)),('unknown tool',lambda:dispatch({'tool':'send_email','args':{}},DATA)),('wrong unit',lambda:run('replay',TRACE_PATH,{**DATA,'unit':'degF'}))]:
    try:call()
    except (AssertionError,ValueError):print('Correctly rejected:',label)
    else:raise AssertionError('Expected rejection: '+label)

## 6. Change, compare, preserve
For practice, alter one temperature and run **live** to obtain new decisions and observations; do not reuse a saved trace under a new input identity. Compare tool order and final values, keeping model revision, task and budget fixed. The constrained menu can limit what changes; do not infer semantic competence merely from a different route. Add another permissible read-only tool only after specifying its contract and adapting termination requirements.

Export the actual trace, exact input, model revision, source and environment versions. `agent-evidence.zip` supports replay of this case; it does not reproduce downloaded model weights or every possible live decision across hardware. Save the notebook separately. No new graded Homework is created.

In [ ]:
import zipfile,hashlib,platform,importlib.metadata as metadata
versions={'python':platform.python_version()}
for package in ['torch','transformers','tokenizers','huggingface-hub']:
    try:versions[package]=metadata.version(package)
    except metadata.PackageNotFoundError:versions[package]='not required for replay'
payload={'input':DATA,'goal':GOAL,'model':MODEL,'revision':REVISION,'execution_mode':MODE,'report':report,'versions':versions}
Path('agent-evidence.json').write_text(json.dumps(payload,indent=2))
with zipfile.ZipFile('agent-evidence.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write('agent-evidence.json');z.write(TRACE_PATH)
print('Saved agent-evidence.zip. Replay validates observations and input identity; it does not run the LLM.')

## Summary
Model policy → validated action → real local tool → observation → next model decision → evidence-backed completion.

Defend: which decisions came from model weights, which were controller constraints, which numbers came from tools, and which evidence would be needed for a broader claim. Explain the live/replay difference without calling replay fresh reasoning.